<a href="https://colab.research.google.com/github/eehujnihs21-stack/app0320/blob/main/2555037%EC%8B%A0%EC%A3%BC%ED%9D%AC_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn httpx beautifulsoup4 gradio pyngrok nest_asyncio pandas deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚠️  여기만 수정!
NGROK_TOKEN = "3DNLGoFzIkfzR8yfEM7o5Zq5pkY_3VWPMLUTgAVrpNsdRZUt4"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import sqlite3, re, threading, time
import pandas as pd
import httpx
import uvicorn
import nest_asyncio
from collections import Counter
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from gradio.routes import mount_gradio_app
from deep_translator import GoogleTranslator
import gradio as gr
from pyngrok import ngrok

nest_asyncio.apply()

# ── DB ───────────────────────────────────────
DB_PATH = "quotes.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DROP TABLE IF EXISTS quotes")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            text       TEXT NOT NULL,
            author     TEXT NOT NULL,
            tags       TEXT,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit(); conn.close()

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

init_db()

# ── 크롤링 ───────────────────────────────────
def crawl_quotes():
    quotes = []
    for page in range(1, 3):
        r    = httpx.get(f"http://quotes.toscrape.com/page/{page}/", timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        for q in soup.select(".quote"):
            quotes.append({
                "text":   q.select_one(".text").get_text(strip=True),
                "author": q.select_one(".author").get_text(strip=True),
                "tags":   ", ".join(t.get_text() for t in q.select(".tag")),
            })
        if len(quotes) >= 20:
            break
    return quotes[:20]

# ── FastAPI ───────────────────────────────────
app = FastAPI(title="격언 관리 API")

class QuoteIn(BaseModel):
    text: str; author: str; tags: str = ""

class QuoteUp(BaseModel):
    text: str | None = None
    author: str | None = None
    tags:   str | None = None

@app.post("/quotes",          summary="격언 추가")
def create(q: QuoteIn):
    conn = get_conn()
    cur  = conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)",
                        (q.text, q.author, q.tags))
    conn.commit(); conn.close()
    return {"id": cur.lastrowid, "message": "저장 완료!"}

@app.get("/quotes",          summary="전체 조회")
def read_all():
    conn = get_conn()
    rows = conn.execute("SELECT * FROM quotes ORDER BY id ASC").fetchall()
    conn.close()
    return [dict(r) for r in rows]

@app.get("/quotes/{qid}",    summary="단건 조회")
def read_one(qid: int):
    conn = get_conn()
    row  = conn.execute("SELECT * FROM quotes WHERE id=?", (qid,)).fetchone()
    conn.close()
    if not row: raise HTTPException(404, "없는 격언입니다")
    return dict(row)

@app.put("/quotes/{qid}",    summary="수정")
def update(qid: int, q: QuoteUp):
    conn = get_conn()
    row  = conn.execute("SELECT * FROM quotes WHERE id=?", (qid,)).fetchone()
    if not row: conn.close(); raise HTTPException(404, "없는 격언입니다")
    conn.execute("UPDATE quotes SET text=?,author=?,tags=? WHERE id=?",
                 (q.text or row["text"], q.author or row["author"],
                  q.tags or row["tags"], qid))
    conn.commit(); conn.close()
    return {"message": "수정 완료!"}

@app.delete("/quotes/{qid}", summary="삭제")
def delete(qid: int):
    conn = get_conn()
    conn.execute("DELETE FROM quotes WHERE id=?", (qid,))
    conn.commit(); conn.close()
    return {"message": "삭제 완료!"}

@app.post("/crawl",          summary="크롤링 실행")
def crawl():
    data = crawl_quotes()
    conn = get_conn()
    conn.execute("DELETE FROM quotes")
    for q in data:
        conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)",
                     (q["text"], q["author"], q["tags"]))
    conn.commit(); conn.close()
    return {"message": f"{len(data)}개 수집 완료!"}

# ── Gradio ────────────────────────────────────
STOP = {"the","a","an","of","in","to","and","is","it","that","for","with",
        "you","i","be","on","are","as","we","this","was","not","or","but",
        "at","by","from","have","had","he","she","they","what","all","when",
        "do","no","so","if","his","her","our","can","will","my","one","who"}

def df_all():
    conn = get_conn()
    rows = conn.execute("SELECT id,author,text,tags FROM quotes ORDER BY id ASC").fetchall()
    conn.close()
    return pd.DataFrame([dict(r) for r in rows]) if rows else pd.DataFrame(columns=["id","author","text","tags"])

def df_word():
    conn = get_conn()
    rows = conn.execute("SELECT text FROM quotes").fetchall()
    conn.close()
    words = [w for r in rows for w in re.findall(r"[a-z]+", r["text"].lower())
             if w not in STOP and len(w) > 3]
    top = Counter(words).most_common(15)
    if not top: return pd.DataFrame({"단어":[],"빈도수":[]})
    w, c = zip(*top)
    return pd.DataFrame({"단어": list(w), "빈도수": list(c)})

def df_author():
    conn = get_conn()
    rows = conn.execute("SELECT author, COUNT(*) as cnt FROM quotes GROUP BY author ORDER BY cnt DESC").fetchall()
    conn.close()
    return pd.DataFrame([dict(r) for r in rows])

def df_tag():
    conn = get_conn()
    rows = conn.execute("SELECT tags FROM quotes WHERE tags!=''").fetchall()
    conn.close()
    tags = [t.strip() for r in rows for t in r["tags"].split(",") if t.strip()]
    top  = Counter(tags).most_common(10)
    if not top: return pd.DataFrame({"태그":[],"빈도수":[]})
    t, c = zip(*top)
    return pd.DataFrame({"태그": list(t), "빈도수": list(c)})

def ui_crawl():
    data = crawl_quotes()
    conn = get_conn()
    conn.execute("DELETE FROM quotes")
    for q in data:
        conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)",
                     (q["text"], q["author"], q["tags"]))
    conn.commit(); conn.close()
    return f"✅ {len(data)}개 수집 완료!"

def ui_add(text, author, tags):
    if not text or not author: return "❌ 내용과 저자는 필수"
    conn = get_conn()
    conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)", (text,author,tags))
    conn.commit(); conn.close()
    return "✅ 추가 완료!"

def ui_delete(qid):
    try:
        conn = get_conn()
        conn.execute("DELETE FROM quotes WHERE id=?", (int(qid),))
        conn.commit(); conn.close()
        return f"✅ ID {qid} 삭제 완료!"
    except Exception as e:
        return f"❌ 오류: {e}"

def load_by_id(qid):
    try:
        conn = get_conn()
        row  = conn.execute("SELECT text FROM quotes WHERE id=?", (int(qid),)).fetchone()
        conn.close()
        if not row: return "❌ 없는 ID예요"
        return row["text"]
    except Exception as e:
        return f"❌ 오류: {e}"

def translate_quote(text):
    if not text: return "❌ 번역할 내용을 입력하세요"
    try:
        return GoogleTranslator(source="en", target="ko").translate(text)
    except Exception as e:
        return f"❌ 번역 오류: {e}"

with gr.Blocks(title="격언 대시보드") as ui:
    gr.Markdown("# 📚 격언 관리 & 분석 대시보드")

    with gr.Tab("🕷️ 크롤링"):
        # 요청하신 '버튼 클릭 → 격언 20개 수집...' 텍스트만 지웠습니다.
        btn = gr.Button("크롤링 실행", variant="primary")
        out = gr.Textbox(label="결과")
        btn.click(ui_crawl, outputs=out)

    with gr.Tab("📋 전체 목록"):
        btn2 = gr.Button("새로고침", variant="primary")
        tbl  = gr.Dataframe()
        btn2.click(df_all, outputs=tbl)

    with gr.Tab("📊 단어 빈도"):
        btn3 = gr.Button("분석", variant="primary")
        plt3 = gr.BarPlot(x="단어", y="빈도수", title="자주 등장하는 단어 Top 15")
        btn3.click(df_word, outputs=plt3)

    with gr.Tab("👤 저자 통계"):
        btn4 = gr.Button("분석", variant="primary")
        tbl4 = gr.Dataframe()
        btn4.click(df_author, outputs=tbl4)

    with gr.Tab("🏷️ 태그 통계"):
        btn5 = gr.Button("분석", variant="primary")
        plt5 = gr.BarPlot(x="태그", y="빈도수", title="인기 태그 Top 10")
        btn5.click(df_tag, outputs=plt5)

    with gr.Tab("✏️ 추가"):
        t1 = gr.Textbox(label="격언 내용")
        t2 = gr.Textbox(label="저자")
        t3 = gr.Textbox(label="태그 (선택)")
        b6 = gr.Button("추가", variant="primary")
        o6 = gr.Textbox(label="결과")
        b6.click(ui_add, inputs=[t1,t2,t3], outputs=o6)

    with gr.Tab("🗑️ 삭제"):
        d1 = gr.Textbox(label="삭제할 격언 ID")
        b7 = gr.Button("삭제", variant="stop")
        o7 = gr.Textbox(label="결과")
        b7.click(ui_delete, inputs=d1, outputs=o7)

    with gr.Tab("🌐 번역"):
        gr.Markdown("격언을 한국어로 번역해드려요!")
        with gr.Row():
            trans_id    = gr.Textbox(label="ID로 불러오기 (선택)")
            load_btn    = gr.Button("불러오기", variant="secondary")
        trans_input = gr.Textbox(label="번역할 격언")
        trans_btn   = gr.Button("번역하기", variant="primary")
        trans_out   = gr.Textbox(label="한국어 번역 결과")
        load_btn.click(load_by_id, inputs=trans_id, outputs=trans_input)
        trans_btn.click(translate_quote, inputs=trans_input, outputs=trans_out)

# ── Gradio → FastAPI 마운트 ───────────────────
app = mount_gradio_app(app, ui, path="/ui")

# ── 서버 백그라운드 실행 ──────────────────────
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
).start()
time.sleep(2)

# ── ngrok 외부 URL ────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
time.sleep(1)
url = ngrok.connect(8000).public_url

print(f"🖥️  Gradio UI  →  {url}/ui")
print(f"📋 Swagger     →  {url}/docs")
print("⛔ 셀 종료하면 서버도 꺼져요! 발표 중엔 유지!")

new /ui
🖥️  Gradio UI  →  https://mop-reclusive-approval.ngrok-free.dev/ui
📋 Swagger     →  https://mop-reclusive-approval.ngrok-free.dev/docs
⛔ 셀 종료하면 서버도 꺼져요! 발표 중엔 유지!
